# 🎗️ Breast Cancer Detection using Machine Learning

This notebook trains a **Random Forest Classifier** to detect whether a breast tumour
is **Malignant** (cancerous) or **Benign** (non-cancerous) based on cell nucleus measurements
extracted from digitized FNA (Fine Needle Aspirate) images.

---
### 📌 Dataset
- **Source:** UCI Machine Learning Repository / `sklearn.datasets.load_breast_cancer()`
- **Samples:** 569 patients
- **Features:** 30 numeric features computed from cell nucleus images
- **Target:** 0 = Malignant, 1 = Benign

### 🧠 Model
- Algorithm: **Random Forest Classifier**
- Why Random Forest? It handles high-dimensional data well, is robust to overfitting,
  and gives feature importances — critical for medical diagnosis.

### 📋 Pipeline
1. Import Libraries
2. Load & Explore Dataset
3. Data Visualization
4. Preprocessing & Train-Test Split
5. Model Training
6. Evaluation (Accuracy, Confusion Matrix, ROC Curve, Classification Report)
7. Feature Importance
8. Save Model

# 1. Import Libraries and Tools

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

# Dataset
from sklearn.datasets import load_breast_cancer

# Preprocessing & Modelling
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

print('All libraries imported successfully ✅')

# 2. Load & Explore Dataset

We use the built-in `sklearn` breast cancer dataset which contains **30 features** derived from
digitized images of fine needle aspirates of breast mass. Each feature describes characteristics
of the cell nuclei present in the image.

**Feature groups:**
- **Mean values** — mean of measurements for each nucleus
- **Standard error** — standard error of measurements
- **Worst values** — mean of the three largest measurements

In [ ]:
# Load dataset
data = load_breast_cancer()

# Create a DataFrame for easier exploration
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print('Dataset Shape:', df.shape)
print('Feature Names:', list(data.feature_names))
print('\nClass Distribution:')
print(df['diagnosis'].value_counts())
print(f'\nMalignant: {(df.target==0).sum()} ({(df.target==0).mean()*100:.1f}%)')
print(f'Benign:    {(df.target==1).sum()} ({(df.target==1).mean()*100:.1f}%)')

In [ ]:
# Preview first 5 rows
df.head()

In [ ]:
# Statistical summary
df.describe().round(3)

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum().sum())
print('No missing values — dataset is clean ✅')

# 3. Data Visualization

Visual exploration helps us understand:
- Class balance between Malignant and Benign
- Which features differ most between classes
- Correlations between features

In [ ]:
# 3.1 Class Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = df['diagnosis'].value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#e74c3c', '#2ecc71'], startangle=90,
            textprops={'fontsize': 13})
axes[0].set_title('Tumour Class Distribution', fontsize=14, fontweight='bold')

# Count bar
sns.countplot(data=df, x='diagnosis', palette={'Malignant':'#e74c3c','Benign':'#2ecc71'}, ax=axes[1])
axes[1].set_title('Count of Each Class', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Diagnosis')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2, p.get_height()+3),
                     ha='center', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# 3.2 Feature distributions: Mean Radius, Mean Texture, Mean Area
key_features = ['mean radius', 'mean texture', 'mean area', 'mean smoothness',
                 'mean compactness', 'mean concavity']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, feat in enumerate(key_features):
    for diagnosis, color in [('Malignant','#e74c3c'), ('Benign','#2ecc71')]:
        subset = df[df['diagnosis'] == diagnosis][feat]
        axes[i].hist(subset, bins=25, alpha=0.6, color=color, label=diagnosis)
    axes[i].set_title(f'{feat.title()} Distribution', fontsize=11)
    axes[i].legend(fontsize=9)
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Count')

plt.suptitle('Feature Distributions by Diagnosis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Correlation heatmap of mean features
mean_cols = [c for c in df.columns if 'mean' in c] + ['target']
corr = df[mean_cols].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap (Mean Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.4 Box plots comparing mean features between Malignant and Benign
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for i, feat in enumerate(key_features):
    sns.boxplot(data=df, x='diagnosis', y=feat,
                palette={'Malignant':'#e74c3c','Benign':'#2ecc71'}, ax=axes[i])
    axes[i].set_title(f'{feat.title()}', fontsize=11)
    axes[i].set_xlabel('')

plt.suptitle('Feature Box Plots by Diagnosis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 4. Preprocessing & Train-Test Split

We apply **StandardScaler** to normalize features — ensuring no single feature
dominates due to scale differences. We use a **80/20 stratified split** to maintain
class proportions in both sets.

In [ ]:
X = data.data
y = data.target

# Stratified split — preserves class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'\nTrain class distribution: {np.bincount(y_train)}')
print(f'Test  class distribution: {np.bincount(y_test)}')

# 5. Model Training

**Random Forest Classifier** — an ensemble of decision trees.
- `n_estimators=200` — 200 trees for robust predictions
- `max_depth=10` — prevents overfitting
- `random_state=42` — reproducibility

We also run **5-fold Stratified Cross-Validation** to get a reliable estimate
of model performance across different data splits.

In [ ]:
# Train Random Forest
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)
print('Model trained successfully ✅')

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
print(f'\n5-Fold CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')
print(f'Individual Fold Scores: {[f"{s*100:.2f}%" for s in cv_scores]}')

# 6. Model Evaluation

We evaluate using four complementary metrics:
- **Accuracy** — overall correct predictions
- **Classification Report** — precision, recall, F1 per class
- **Confusion Matrix** — true/false positives and negatives
- **ROC-AUC Curve** — discriminative ability across thresholds

In [ ]:
# Predictions
y_pred      = model.predict(X_test_scaled)
y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc_score = roc_auc_score(y_test, y_pred_prob)

print(f'Test Accuracy : {acc*100:.2f}%')
print(f'ROC-AUC Score : {auc_score:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Malignant','Benign']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malignant','Benign'],
            yticklabels=['Malignant','Benign'],
            linewidths=1, cbar=False)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc     = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='#e74c3c', lw=2.5,
         label=f'ROC Curve (AUC = {roc_auc:.3f})')
plt.plot([0,1],[0,1], color='gray', linestyle='--', lw=1.5, label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Breast Cancer Detection', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 7. Feature Importance

Random Forest provides built-in **feature importance scores** — how much each
feature contributes to the model's decisions. This is clinically valuable:
it tells doctors which measurements are most predictive.

In [ ]:
importances = model.feature_importances_
feat_names  = data.feature_names
sorted_idx  = np.argsort(importances)[::-1]

# Top 15 features
top_n = 15
plt.figure(figsize=(11, 6))
colors = ['#e74c3c' if i < 5 else '#3498db' if i < 10 else '#95a5a6'
          for i in range(top_n)]
plt.bar(range(top_n), importances[sorted_idx[:top_n]], color=colors, edgecolor='white')
plt.xticks(range(top_n), [feat_names[i] for i in sorted_idx[:top_n]],
           rotation=45, ha='right', fontsize=9)
plt.title('Top 15 Feature Importances — Random Forest', fontsize=14, fontweight='bold')
plt.ylabel('Importance Score', fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Top 5 most important features:')
for i in range(5):
    print(f'  {i+1}. {feat_names[sorted_idx[i]]:<35} {importances[sorted_idx[i]]:.4f}')

# 8. Save Model

We save the trained model using `pickle` so it can be loaded directly
into the **Streamlit web app** without retraining.

> ⚠️ Save to `saved_models/breast_cancer_model.sav` — this path is expected by `main.py`.

In [ ]:
import os
os.makedirs('saved_models', exist_ok=True)

# Save the trained model
with open('saved_models/breast_cancer_model.sav', 'wb') as f:
    pickle.dump(model, f)

print('✅ Model saved to: saved_models/breast_cancer_model.sav')

In [ ]:
# Verify: reload and test
loaded_model = pickle.load(open('saved_models/breast_cancer_model.sav', 'rb'))
sample       = X_test_scaled[0].reshape(1, -1)
prediction   = loaded_model.predict(sample)
label_map    = {0: 'Malignant', 1: 'Benign'}

print(f'Sample prediction : {label_map[prediction[0]]}')
print(f'Actual label      : {label_map[y_test[0]]}')
print('\n🎉 Model loaded and working correctly!')